# The Disordered Wall — cleaned notebook

This notebook streamlines the final analysis for the **ProteinGym × DisProt** project.

## Core question
Do mutation effects differ between **intrinsically disordered regions (IDRs)** and **ordered regions**, and are IDR mutations harder to model with a simple supervised baseline?

## Notebook flow
1. Setup and paths  
2. Core helper functions  
3. Build the intersected benchmark  
4. Summarize region-level mutation effects  
5. Train and compare baseline models  
6. Add robustness checks and follow-up analyses  

> **Note:** In the current pipeline, structure-derived features may remain unavailable if the ProteinGym AF structure mapping does not populate. The main analysis below is still valid for the mutation-only and mutation-plus-region comparisons.


## 1. Setup and file paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install biopython pandas numpy scikit-learn scipy matplotlib requests

from pathlib import Path
from types import SimpleNamespace

BASE = Path("/content/drive/MyDrive")

args = SimpleNamespace(
    reference_csv=BASE / "DMS_substitutions.csv",
    assay_dir=BASE / "DMS_ProteinGym_substitutions",
    disprot_file=BASE / "DisProt_release_2025_12.tsv",
    structure_dir=BASE / "ProteinGym_AF2_structures",
    project_root=BASE / "disordered_wall_project",
    max_common_proteins=-1,
    min_variants_per_assay=10,
    max_assays_per_protein=5,
    seq_window=5,
    struct_radius=10.0,
    n_splits=5,
    target_column="DMS_score_z",
    group_column="UniProt_ID",
)

print(args)


## 2. Core imports and utility functions

In [ ]:
# --- Imports, constants, path helpers, and general utilities ---

import os
import re
import json
import math
import time
from pathlib import Path
from types import SimpleNamespace
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from Bio.PDB import PDBParser, MMCIFParser
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold


AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AA3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    "SEC": "U", "PYL": "O",
}

HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

VOLUME = {
    "A": 88.6, "C": 108.5, "D": 111.1, "E": 138.4, "F": 189.9,
    "G": 60.1, "H": 153.2, "I": 166.7, "K": 168.6, "L": 166.7,
    "M": 162.9, "N": 114.1, "P": 112.7, "Q": 143.8, "R": 173.4,
    "S": 89.0, "T": 116.1, "V": 140.0, "W": 227.8, "Y": 193.6,
}

CHARGE_CLASS = {
    "D": -1, "E": -1,
    "K": 1, "R": 1, "H": 0.5,
}

AROMATIC = set("FWYH")
POLAR = set("STNQCYWHKRED")
SMALL = set("AGSTCPDNV")

DEFAULT_TARGET = "DMS_score_z"
DEFAULT_GROUP = "UniProt_ID"
DEFAULT_N_SPLITS = 5
DEFAULT_RANDOM_STATE = 42
DEFAULT_SEQ_WINDOW = 5
DEFAULT_STRUCT_RADIUS = 10.0

# parse_args removed for notebook version

def ensure_dirs(project_root: Path) -> Dict[str, Path]:
    output_dir = project_root / "outputs"
    fig_dir = output_dir / "figures"
    table_dir = output_dir / "tables"
    cache_dir = project_root / "cache"
    for p in [project_root, output_dir, fig_dir, table_dir, cache_dir]:
        p.mkdir(parents=True, exist_ok=True)
    return {
        "project_root": project_root,
        "output_dir": output_dir,
        "fig_dir": fig_dir,
        "table_dir": table_dir,
        "cache_dir": cache_dir,
    }

def normalize_uniprot_id(x: object) -> Optional[str]:
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    s = s.split(";")[0].strip()
    s = s.split("-")[0].strip()  # collapse isoforms to canonical accession for matching
    return s.upper()



def detect_column(df: pd.DataFrame, candidates: Iterable[str], required: bool = True) -> Optional[str]:
    normalized = {c.lower().strip(): c for c in df.columns}
    normalized_nospace = {re.sub(r"[^a-z0-9]+", "", c.lower()): c for c in df.columns}

    for cand in candidates:
        c0 = cand.lower().strip()
        c1 = re.sub(r"[^a-z0-9]+", "", cand.lower())
        if c0 in normalized:
            return normalized[c0]
        if c1 in normalized_nospace:
            return normalized_nospace[c1]

    if required:
        raise KeyError(
            f"Could not find any of these columns: {list(candidates)}\nAvailable columns: {list(df.columns)}"
        )
    return None



def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path}")



def safe_zscore_by_group(series: pd.Series, groups: pd.Series) -> pd.Series:
    def _z(x: pd.Series) -> pd.Series:
        sd = x.std(ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(np.zeros(len(x)), index=x.index)
        return (x - x.mean()) / sd

    return series.groupby(groups).transform(_z)



def parse_single_mutant(mut: str) -> Optional[Tuple[str, int, str]]:
    """Parse e.g. A123V -> ('A', 123, 'V')."""
    if pd.isna(mut):
        return None
    m = re.fullmatch(r"([A-Z])(\d+)([A-Z])", str(mut).strip())
    if not m:
        return None
    wt, pos, mt = m.group(1), int(m.group(2)), m.group(3)
    return wt, pos, mt



def find_first_existing(paths: Iterable[Path]) -> Optional[Path]:
    for p in paths:
        if p and p.exists():
            return p
    return None

## 3. Data loading, overlap construction, and feature engineering

In [ ]:
# --- DisProt loading ---

def flatten_disprot_json(obj: object, records: List[Dict]) -> None:
    """Recursively extract region-like DisProt records from flexible JSON exports."""
    if isinstance(obj, dict):
        entry_uniprot = None
        for key in ["acc", "accession", "uniprot_acc", "uniprot", "uniprot_id", "uniprotAccession"]:
            if key in obj and obj[key]:
                entry_uniprot = normalize_uniprot_id(obj[key])
                break

        entry_disprot = None
        for key in ["disprot_id", "disprotId", "id"]:
            if key in obj and isinstance(obj[key], str) and obj[key].upper().startswith("DP"):
                entry_disprot = obj[key]
                break

        if "regions" in obj and isinstance(obj["regions"], list):
            for region in obj["regions"]:
                if not isinstance(region, dict):
                    continue
                start = region.get("start") or region.get("start_position") or region.get("begin")
                end = region.get("end") or region.get("end_position") or region.get("stop")
                if start is not None and end is not None:
                    records.append(
                        {
                            "UniProt_ID": entry_uniprot,
                            "DisProt_ID": entry_disprot,
                            "start": int(start),
                            "end": int(end),
                        }
                    )
        for v in obj.values():
            flatten_disprot_json(v, records)

    elif isinstance(obj, list):
        for x in obj:
            flatten_disprot_json(x, records)



def load_disprot_regions(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"DisProt file not found: {path}")

    if path.suffix.lower() == ".json":
        with open(path, "r") as f:
            obj = json.load(f)
        records: List[Dict] = []
        flatten_disprot_json(obj, records)
        regions = pd.DataFrame(records)
    else:
        raw = read_table(path)
        uniprot_col = detect_column(
            raw,
            [
                "UniProt_ID", "UniProt ACC", "uniprot_acc", "uniprot", "accession",
                "acc", "uniprotid", "UniProt accession"
            ],
        )
        start_col = detect_column(raw, ["start", "start_position", "begin", "region_start"])
        end_col = detect_column(raw, ["end", "end_position", "stop", "region_end"])
        disprot_col = detect_column(raw, ["DisProt_ID", "disprot_id", "id"], required=False)

        regions = pd.DataFrame(
            {
                "UniProt_ID": raw[uniprot_col].map(normalize_uniprot_id),
                "start": pd.to_numeric(raw[start_col], errors="coerce"),
                "end": pd.to_numeric(raw[end_col], errors="coerce"),
                "DisProt_ID": raw[disprot_col] if disprot_col else None,
            }
        )

    regions = regions.dropna(subset=["UniProt_ID", "start", "end"]).copy()
    regions["start"] = regions["start"].astype(int)
    regions["end"] = regions["end"].astype(int)
    regions = regions.query("start >= 1 and end >= start").drop_duplicates()
    return regions



def build_idr_masks(reference_df: pd.DataFrame, disprot_regions: pd.DataFrame) -> Dict[str, np.ndarray]:
    mask_dict: Dict[str, np.ndarray] = {}
    ref_small = reference_df[["UniProt_ID", "target_seq"]].drop_duplicates()
    region_groups = disprot_regions.groupby("UniProt_ID")

    for _, row in ref_small.iterrows():
        uid = row["UniProt_ID"]
        seq = row["target_seq"]
        if not isinstance(seq, str) or not seq:
            continue
        mask = np.zeros(len(seq), dtype=np.int8)
        if uid in region_groups.groups:
            sub = region_groups.get_group(uid)
            for _, rr in sub.iterrows():
                start0 = max(0, int(rr["start"]) - 1)
                end0 = min(len(seq), int(rr["end"]))
                mask[start0:end0] = 1
        mask_dict[uid] = mask
    return mask_dict

# --- ProteinGym loading (base helpers) ---

def load_proteingym_reference(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"ProteinGym reference CSV not found: {path}")

    ref = read_table(path)
    dms_col = detect_column(ref, ["DMS_id", "DMS ID", "assay_id", "DMS filename"])
    uid_col = detect_column(ref, ["UniProt_ID", "uniprot_id", "UniProt ID", "UniProt", "uniprot"])
    seq_col = detect_column(ref, ["target_seq", "target sequence", "sequence", "wt_sequence"])

    out = ref.copy()
    out["DMS_id"] = out[dms_col].astype(str)
    out["UniProt_ID"] = out[uid_col].map(normalize_uniprot_id)
    out["target_seq"] = out[seq_col].astype(str)

    keep_cols = ["DMS_id", "UniProt_ID", "target_seq"]
    for extra in ["MSA_depth_category", "selection_type", "Taxa", "taxon"]:
        c = detect_column(out, [extra], required=False)
        if c and c not in keep_cols:
            keep_cols.append(c)

    return out[keep_cols].dropna(subset=["DMS_id", "UniProt_ID", "target_seq"]).drop_duplicates()



def match_assay_file(dms_id: str, assay_dir: Path) -> Optional[Path]:
    candidates = list(assay_dir.glob(f"{dms_id}.csv"))
    if candidates:
        return candidates[0]

    candidates = list(assay_dir.glob(f"{dms_id}*.csv"))
    if candidates:
        return candidates[0]

    for p in assay_dir.glob("*.csv"):
        if dms_id in p.name:
            return p
    return None



def load_single_assay(path: Path, dms_id: str) -> pd.DataFrame:
    df = read_table(path)
    mutant_col = detect_column(df, ["mutant", "mutation", "variant"])
    score_col = detect_column(df, ["DMS_score", "score", "fitness", "dms_score"])
    score_bin_col = detect_column(df, ["DMS_score_bin", "score_bin", "label", "fitness_bin"], required=False)
    seq_col = detect_column(df, ["mutated_sequence", "mut_seq", "sequence_mutant"], required=False)

    out = pd.DataFrame(
        {
            "DMS_id": dms_id,
            "mutant": df[mutant_col].astype(str),
            "DMS_score": pd.to_numeric(df[score_col], errors="coerce"),
        }
    )
    if score_bin_col:
        out["DMS_score_bin"] = pd.to_numeric(df[score_bin_col], errors="coerce")
    if seq_col:
        out["mutated_sequence"] = df[seq_col].astype(str)

    return out.dropna(subset=["mutant", "DMS_score"])



def build_intersected_assay_table(
    reference_df: pd.DataFrame,
    disprot_regions: pd.DataFrame,
    assay_dir: Path,
    max_common_proteins: Optional[int],
    min_variants_per_assay: int,
    max_assays_per_protein: int,
) -> pd.DataFrame:
    if not assay_dir.exists():
        raise FileNotFoundError(f"ProteinGym assay directory not found: {assay_dir}")

    common_uids = sorted(set(reference_df["UniProt_ID"]) & set(disprot_regions["UniProt_ID"]))
    if max_common_proteins is not None and max_common_proteins > 0:
        common_uids = common_uids[:max_common_proteins]

    ref_sub = reference_df[reference_df["UniProt_ID"].isin(common_uids)].copy()
    ref_sub["assay_rank_within_protein"] = ref_sub.groupby("UniProt_ID").cumcount()
    ref_sub = ref_sub[ref_sub["assay_rank_within_protein"] < max_assays_per_protein].copy()

    assay_tables = []
    missing_files = []

    for _, rr in ref_sub.iterrows():
        dms_id = rr["DMS_id"]
        assay_file = match_assay_file(dms_id, assay_dir)
        if assay_file is None:
            missing_files.append(dms_id)
            continue
        try:
            assay_tables.append(load_single_assay(assay_file, dms_id))
        except Exception as e:
            print(f"[WARN] Failed to load assay {dms_id}: {e}")

    if not assay_tables:
        raise RuntimeError("No ProteinGym assay files could be loaded. Check your assay directory and file naming.")

    assays = pd.concat(assay_tables, ignore_index=True)
    merged = assays.merge(ref_sub.drop(columns=["assay_rank_within_protein"]), on="DMS_id", how="inner")

    if missing_files:
        print(f"[INFO] Missing assay files for {len(missing_files)} reference rows")

    parsed = merged["mutant"].map(parse_single_mutant)
    merged = merged[parsed.notna()].copy()
    merged[["wt_aa", "position", "mut_aa"]] = pd.DataFrame(parsed[parsed.notna()].tolist(), index=merged.index)

    merged = merged[merged["wt_aa"].isin(AA_LIST) & merged["mut_aa"].isin(AA_LIST)].copy()
    merged = merged[merged["position"] >= 1].copy()
    merged = merged[merged["position"] <= merged["target_seq"].str.len()].copy()

    def wt_matches(row) -> bool:
        seq = row["target_seq"]
        pos = int(row["position"])
        return isinstance(seq, str) and len(seq) >= pos and seq[pos - 1] == row["wt_aa"]

    merged["wt_matches_target"] = merged.apply(wt_matches, axis=1)
    merged = merged[merged["wt_matches_target"]].copy()

    merged["DMS_score_z"] = safe_zscore_by_group(merged["DMS_score"], merged["DMS_id"])

    assay_sizes = merged.groupby("DMS_id").size()
    keep_assays = assay_sizes[assay_sizes >= min_variants_per_assay].index
    merged = merged[merged["DMS_id"].isin(keep_assays)].copy()

    return merged.reset_index(drop=True)

# --- Final patched ProteinGym/UniProt mapping used in this notebook ---

import time
import requests

# UniProt accession regex (canonical + isoforms)
_ACC_RE = re.compile(
    r"^(?:"
    r"[OPQ][0-9][A-Z0-9]{3}[0-9]"
    r"|"
    r"[A-NR-Z][0-9][A-Z0-9]{3}[0-9]"
    r")(?:-\d+)?$"
)

def normalize_uniprot_accession(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if not s:
        return None
    # strip isoform suffix for overlap with DisProt
    s = s.split("-")[0]
    return s

def looks_like_accession(x):
    if pd.isna(x):
        return False
    return bool(_ACC_RE.fullmatch(str(x).strip().upper()))

def entry_name_to_accession(name, sleep_s=0.05):
    """
    Convert a ProteinGym UniProt_ID value (often an entry name like ACE2_HUMAN)
    into a canonical accession like Q9BYF1.
    """
    if pd.isna(name):
        return None

    s = str(name).strip().upper()
    if not s:
        return None

    # Already an accession
    if looks_like_accession(s):
        return normalize_uniprot_accession(s)

    # Common ProteinGym case: ACCESSION_SPECIES, e.g. A0A140D2T1_ZIKV
    prefix = s.split("_")[0]
    if looks_like_accession(prefix):
        return normalize_uniprot_accession(prefix)

    # Otherwise query UniProt by entry name / ID
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"id:{s}",
        "fields": "accession,id",
        "format": "json",
        "size": 1,
    }

    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        js = r.json()
        results = js.get("results", [])
        if results:
            acc = results[0].get("primaryAccession")
            return normalize_uniprot_accession(acc)
    except Exception:
        pass

    time.sleep(sleep_s)
    return None


def build_proteingym_id_map(entry_names):
    uniq = sorted({str(x).strip() for x in entry_names if pd.notna(x) and str(x).strip()})
    mapping = {}
    for i, name in enumerate(uniq, start=1):
        mapping[name] = entry_name_to_accession(name)
        if i % 25 == 0 or i == len(uniq):
            print(f"Mapped {i}/{len(uniq)} ProteinGym UniProt IDs")
    return mapping


def load_proteingym_reference(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"ProteinGym reference CSV not found: {path}")

    ref = read_table(path)

    dms_id_col = detect_column(ref, ["DMS_id", "DMS ID", "assay_id"])
    dms_file_col = detect_column(ref, ["DMS_filename", "DMS filename", "filename"], required=False)
    uid_col = detect_column(ref, ["UniProt_ID", "uniprot_id", "UniProt ID", "UniProt", "uniprot"])
    seq_col = detect_column(ref, ["target_seq", "target sequence", "sequence", "wt_sequence"])

    out = ref.copy()
    out["DMS_id"] = out[dms_id_col].astype(str).str.strip()

    if dms_file_col:
        out["DMS_filename"] = out[dms_file_col].astype(str).str.strip()
    else:
        out["DMS_filename"] = out["DMS_id"] + ".csv"

    # Keep original ProteinGym identifier for debugging
    out["UniProt_entry_name"] = out[uid_col].astype(str).str.strip()

    # Map to accessions
    id_map = build_proteingym_id_map(out["UniProt_entry_name"].unique())
    out["UniProt_ID"] = out["UniProt_entry_name"].map(id_map).map(normalize_uniprot_accession)

    out["target_seq"] = out[seq_col].astype(str).str.strip()

    print("\nProteinGym UniProt mapping summary:")
    print("  total rows:", len(out))
    print("  mapped rows:", int(out["UniProt_ID"].notna().sum()))
    print("  unique mapped accessions:", int(out["UniProt_ID"].nunique()))

    unmapped = sorted(out.loc[out["UniProt_ID"].isna(), "UniProt_entry_name"].dropna().unique().tolist())
    print("  first 20 unmapped entry names:", unmapped[:20])

    keep_cols = [
        "DMS_id",
        "DMS_filename",
        "UniProt_ID",
        "UniProt_entry_name",
        "target_seq",
    ]

    return (
        out[keep_cols]
        .dropna(subset=["DMS_id", "UniProt_ID", "target_seq"])
        .drop_duplicates()
        .reset_index(drop=True)
    )

# --- Final patched assay matching and overlap logic used in this notebook ---

def load_proteingym_reference(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"ProteinGym reference CSV not found: {path}")

    ref = read_table(path)
    dms_id_col = detect_column(ref, ["DMS_id", "DMS ID", "assay_id"])
    dms_file_col = detect_column(ref, ["DMS_filename", "DMS filename", "filename"], required=False)
    uid_col = detect_column(ref, ["UniProt_ID", "uniprot_id", "UniProt ID", "UniProt", "uniprot"])
    seq_col = detect_column(ref, ["target_seq", "target sequence", "sequence", "wt_sequence"])

    out = ref.copy()
    out["DMS_id"] = out[dms_id_col].astype(str).str.strip()
    if dms_file_col:
        out["DMS_filename"] = out[dms_file_col].astype(str).str.strip()
    else:
        out["DMS_filename"] = out["DMS_id"] + ".csv"

    out["UniProt_ID"] = out[uid_col].map(normalize_uniprot_id)
    out["target_seq"] = out[seq_col].astype(str)

    keep_cols = ["DMS_id", "DMS_filename", "UniProt_ID", "target_seq"]
    for extra in ["MSA_depth_category", "selection_type", "Taxa", "taxon"]:
        c = detect_column(out, [extra], required=False)
        if c and c not in keep_cols:
            keep_cols.append(c)

    return (
        out[keep_cols]
        .dropna(subset=["DMS_id", "UniProt_ID", "target_seq"])
        .drop_duplicates()
    )


def match_assay_file(dms_id: str, assay_dir: Path, dms_filename: Optional[str] = None) -> Optional[Path]:
    def _norm_name(x: str) -> str:
        s = Path(str(x)).stem.lower().strip()
        s = re.sub(r"\.csv$", "", s)
        s = re.sub(r"[^a-z0-9]+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        return s

    direct_candidates = []

    if dms_filename is not None and str(dms_filename).strip() and str(dms_filename).lower() != "nan":
        fn = str(dms_filename).strip()
        direct_candidates.append(assay_dir / fn)
        if not fn.lower().endswith(".csv"):
            direct_candidates.append(assay_dir / f"{fn}.csv")

    did = str(dms_id).strip()
    direct_candidates.append(assay_dir / f"{did}.csv")

    for p in direct_candidates:
        if p.exists():
            return p

    all_csvs = list(assay_dir.glob("*.csv"))
    if not all_csvs:
        return None

    targets = {_norm_name(did)}
    if dms_filename is not None and str(dms_filename).strip() and str(dms_filename).lower() != "nan":
        targets.add(_norm_name(str(dms_filename)))

    # exact normalized match
    for p in all_csvs:
        if _norm_name(p.name) in targets or _norm_name(p.stem) in targets:
            return p

    # substring fallback
    for p in all_csvs:
        pn = _norm_name(p.name)
        if any(t in pn or pn in t for t in targets):
            return p

    return None


def build_intersected_assay_table(
    reference_df: pd.DataFrame,
    disprot_regions: pd.DataFrame,
    assay_dir: Path,
    max_common_proteins: Optional[int],
    min_variants_per_assay: int,
    max_assays_per_protein: int,
) -> pd.DataFrame:
    if not assay_dir.exists():
        raise FileNotFoundError(f"ProteinGym assay directory not found: {assay_dir}")

    common_uids = sorted(set(reference_df["UniProt_ID"]) & set(disprot_regions["UniProt_ID"]))
    print(f"Common UniProt proteins between ProteinGym and DisProt: {len(common_uids)}")

    if len(common_uids) == 0:
        raise RuntimeError(
            "ProteinGym and DisProt have zero UniProt overlap after normalization. "
            "This usually means the DisProt file format/columns were parsed wrong."
        )

    ref_sub = reference_df[reference_df["UniProt_ID"].isin(common_uids)].copy()
    print(f"Reference rows in overlap: {len(ref_sub)}")

    # match files FIRST, before limiting proteins
    ref_sub["assay_file"] = ref_sub.apply(
        lambda r: match_assay_file(
            dms_id=r["DMS_id"],
            assay_dir=assay_dir,
            dms_filename=r["DMS_filename"] if "DMS_filename" in ref_sub.columns else None,
        ),
        axis=1,
    )

    matched_ref = ref_sub[ref_sub["assay_file"].notna()].copy()
    unmatched_ref = ref_sub[ref_sub["assay_file"].isna()].copy()

    print(f"Overlapping reference rows with assay files present: {len(matched_ref)}")
    print(f"Overlapping reference rows missing assay files: {len(unmatched_ref)}")

    if len(matched_ref) == 0:
        print("Example missing DMS IDs:")
        print(ref_sub["DMS_id"].head(20).tolist())
        raise RuntimeError(
            "There is UniProt overlap, but none of those overlapping assays were found in your assay folder."
        )

    # keep proteins with the most matched assays
    protein_counts = (
        matched_ref.groupby("UniProt_ID")["DMS_id"]
        .count()
        .sort_values(ascending=False)
    )
    keep_uids = protein_counts.index.tolist()

    if max_common_proteins is not None and max_common_proteins > 0:
        keep_uids = keep_uids[:max_common_proteins]

    matched_ref = matched_ref[matched_ref["UniProt_ID"].isin(keep_uids)].copy()

    matched_ref["assay_rank_within_protein"] = matched_ref.groupby("UniProt_ID").cumcount()
    matched_ref = matched_ref[matched_ref["assay_rank_within_protein"] < max_assays_per_protein].copy()

    assay_tables = []
    failed_loads = []

    for _, rr in matched_ref.iterrows():
        dms_id = rr["DMS_id"]
        assay_file = rr["assay_file"]
        try:
            assay_df = load_single_assay(Path(assay_file), dms_id)
            if len(assay_df) >= min_variants_per_assay:
                assay_tables.append(assay_df)
        except Exception as e:
            failed_loads.append((dms_id, str(e)))

    print(f"Assay tables successfully loaded: {len(assay_tables)}")
    if failed_loads:
        print("First few assay load failures:")
        print(failed_loads[:10])

    if not assay_tables:
        raise RuntimeError(
            "Assay files were found, but none could be parsed into usable mutation tables."
        )

    assays = pd.concat(assay_tables, ignore_index=True)
    merged = assays.merge(
        matched_ref.drop(columns=["assay_rank_within_protein", "assay_file"]),
        on="DMS_id",
        how="inner",
    )

    parsed = merged["mutant"].map(parse_single_mutant)
    merged = merged[parsed.notna()].copy()
    merged[["wt_aa", "position", "mut_aa"]] = pd.DataFrame(parsed[parsed.notna()].tolist(), index=merged.index)

    merged["position"] = merged["position"].astype(int)
    merged["seq_len"] = merged["target_seq"].str.len().astype(int)
    merged = merged[(merged["position"] >= 1) & (merged["position"] <= merged["seq_len"])].copy()

    merged["seq_wt_aa"] = merged.apply(lambda r: r["target_seq"][r["position"] - 1], axis=1)
    merged["wt_match"] = (merged["wt_aa"] == merged["seq_wt_aa"]).astype(int)
    merged = merged[merged["wt_match"] == 1].copy()

    merged["DMS_score_z"] = safe_zscore_by_group(merged["DMS_score"], merged["DMS_id"])
    merged["position_frac"] = merged["position"] / merged["seq_len"]

    return merged.reset_index(drop=True)

# --- Region and mutation features ---

def local_mean(arr: np.ndarray, center_pos_1based: int, window: int) -> float:
    idx = center_pos_1based - 1
    lo = max(0, idx - window)
    hi = min(len(arr), idx + window + 1)
    if hi <= lo:
        return np.nan
    return float(np.mean(arr[lo:hi]))



def add_region_features(df: pd.DataFrame, idr_masks: Dict[str, np.ndarray], seq_window: int) -> pd.DataFrame:
    out = df.copy()
    region_type = []
    is_disordered = []
    local_disorder_density = []
    seq_len = []
    pos_frac = []

    for _, row in out.iterrows():
        uid = row["UniProt_ID"]
        pos = int(row["position"])
        seq = row["target_seq"]
        mask = idr_masks.get(uid)

        seq_len.append(len(seq))
        pos_frac.append(pos / len(seq) if len(seq) > 0 else np.nan)

        if mask is None or pos < 1 or pos > len(mask):
            is_disordered.append(np.nan)
            region_type.append("unknown")
            local_disorder_density.append(np.nan)
            continue

        flag = int(mask[pos - 1])
        is_disordered.append(flag)
        region_type.append("IDR" if flag == 1 else "ordered")
        local_disorder_density.append(local_mean(mask.astype(float), pos, seq_window))

    out["sequence_length"] = seq_len
    out["position_frac"] = pos_frac
    out["is_disordered"] = is_disordered
    out["region_type"] = region_type
    out["local_disorder_density"] = local_disorder_density
    return out



def add_mutation_descriptor_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["hydropathy_wt"] = out["wt_aa"].map(HYDROPATHY)
    out["hydropathy_mut"] = out["mut_aa"].map(HYDROPATHY)
    out["delta_hydropathy"] = out["hydropathy_mut"] - out["hydropathy_wt"]

    out["volume_wt"] = out["wt_aa"].map(VOLUME)
    out["volume_mut"] = out["mut_aa"].map(VOLUME)
    out["delta_volume"] = out["volume_mut"] - out["volume_wt"]

    out["charge_wt"] = out["wt_aa"].map(CHARGE_CLASS).fillna(0)
    out["charge_mut"] = out["mut_aa"].map(CHARGE_CLASS).fillna(0)
    out["delta_charge"] = out["charge_mut"] - out["charge_wt"]

    out["wt_is_aromatic"] = out["wt_aa"].isin(AROMATIC).astype(int)
    out["mut_is_aromatic"] = out["mut_aa"].isin(AROMATIC).astype(int)
    out["wt_is_polar"] = out["wt_aa"].isin(POLAR).astype(int)
    out["mut_is_polar"] = out["mut_aa"].isin(POLAR).astype(int)
    out["wt_is_small"] = out["wt_aa"].isin(SMALL).astype(int)
    out["mut_is_small"] = out["mut_aa"].isin(SMALL).astype(int)
    out["is_conservative_substitution"] = (
        (out["wt_is_aromatic"] == out["mut_is_aromatic"]) &
        (out["wt_is_polar"] == out["mut_is_polar"]) &
        (np.sign(out["charge_wt"]) == np.sign(out["charge_mut"]))
    ).astype(int)

    return out

# --- Optional structure features ---

@dataclass
class ResidueStructFeature:
    residue_index: int
    residue_name: str
    plddt: float
    ca_neighbors_10A: float
    local_plddt_seq_window: float



def find_structure_file(uniprot_id: str, structure_dir: Optional[Path]) -> Optional[Path]:
    if structure_dir is None or not structure_dir.exists():
        return None

    patterns = [
        f"*{uniprot_id}*.pdb",
        f"*{uniprot_id}*.cif",
        f"*{uniprot_id}*.mmcif",
    ]
    matches: List[Path] = []
    for pat in patterns:
        matches.extend(structure_dir.glob(pat))
    if not matches:
        return None
    return sorted(matches)[0]



def load_structure_residue_features(structure_file: Path, seq_window: int, struct_radius: float) -> Dict[int, ResidueStructFeature]:
    suffix = structure_file.suffix.lower()
    parser = MMCIFParser(QUIET=True) if suffix in {".cif", ".mmcif"} else PDBParser(QUIET=True)
    structure = parser.get_structure("model", str(structure_file))

    residues: List[Tuple[int, str, np.ndarray, float]] = []
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, resseq, icode = residue.id
                if hetflag.strip():
                    continue
                resname = residue.get_resname().upper()
                aa1 = AA3_TO_1.get(resname)
                if aa1 is None or "CA" not in residue:
                    continue
                ca = residue["CA"]
                coord = np.array(ca.coord, dtype=float)
                b = float(ca.get_bfactor())  # AF-style pLDDT often stored in B-factor
                residues.append((int(resseq), aa1, coord, b))
        break

    if not residues:
        return {}

    residues.sort(key=lambda x: x[0])
    positions = np.array([r[0] for r in residues], dtype=int)
    aas = [r[1] for r in residues]
    coords = np.vstack([r[2] for r in residues])
    plddts = np.array([r[3] for r in residues], dtype=float)

    dmat = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2))
    neighbor_counts = (dmat <= struct_radius).sum(axis=1) - 1

    features: Dict[int, ResidueStructFeature] = {}
    for i, pos in enumerate(positions):
        lo = max(0, i - seq_window)
        hi = min(len(positions), i + seq_window + 1)
        features[int(pos)] = ResidueStructFeature(
            residue_index=int(pos),
            residue_name=aas[i],
            plddt=float(plddts[i]),
            ca_neighbors_10A=float(neighbor_counts[i]),
            local_plddt_seq_window=float(np.mean(plddts[lo:hi])),
        )
    return features



def build_structure_feature_cache(
    df: pd.DataFrame,
    structure_dir: Optional[Path],
    seq_window: int,
    struct_radius: float,
) -> Dict[str, Dict[int, ResidueStructFeature]]:
    cache: Dict[str, Dict[int, ResidueStructFeature]] = {}
    for uid in sorted(df["UniProt_ID"].dropna().unique()):
        f = find_structure_file(uid, structure_dir)
        if f is None:
            cache[uid] = {}
            continue
        try:
            cache[uid] = load_structure_residue_features(f, seq_window=seq_window, struct_radius=struct_radius)
        except Exception as e:
            print(f"[WARN] Failed structure parsing for {uid}: {e}")
            cache[uid] = {}
    return cache



def add_structure_features(df: pd.DataFrame, structure_cache: Dict[str, Dict[int, ResidueStructFeature]]) -> pd.DataFrame:
    out = df.copy()
    plddt = []
    ca_neighbors = []
    local_plddt = []
    structure_available = []
    structure_wt_match = []

    for _, row in out.iterrows():
        uid = row["UniProt_ID"]
        pos = int(row["position"])
        wt = row["wt_aa"]
        feats = structure_cache.get(uid, {})
        rf = feats.get(pos)
        if rf is None:
            plddt.append(np.nan)
            ca_neighbors.append(np.nan)
            local_plddt.append(np.nan)
            structure_available.append(0)
            structure_wt_match.append(np.nan)
            continue

        plddt.append(rf.plddt)
        ca_neighbors.append(rf.ca_neighbors_10A)
        local_plddt.append(rf.local_plddt_seq_window)
        structure_available.append(1)
        structure_wt_match.append(1 if rf.residue_name == wt else 0)

    out["structure_available"] = structure_available
    out["plddt"] = plddt
    out["local_plddt"] = local_plddt
    out["ca_neighbors_10A"] = ca_neighbors
    out["structure_wt_match"] = structure_wt_match
    return out

## 4. Modeling and plotting helpers

In [ ]:
# --- Modeling helpers ---

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))



def spearman_safe(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 2:
        return np.nan
    r, _ = spearmanr(y_true, y_pred)
    return float(r) if np.isfinite(r) else np.nan



def make_model_pipeline(categorical_cols: List[str], numeric_cols: List[str], random_state: int) -> Pipeline:
    pre = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_cols,
            ),
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    model = RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=random_state,
    )

    return Pipeline([("preprocess", pre), ("model", model)])



def evaluate_feature_set(
    df: pd.DataFrame,
    feature_name: str,
    categorical_cols: List[str],
    numeric_cols: List[str],
    target_col: str,
    group_col: str,
    n_splits: int,
    random_state: int,
) -> pd.DataFrame:
    df = df.loc[:, ~pd.Index(df.columns).duplicated()].copy()

    use_cols = categorical_cols + numeric_cols + [target_col, group_col, "region_type"]
    use_cols = list(dict.fromkeys(use_cols))  # preserve order, remove duplicates

    data = df[use_cols].copy()
    data = data.dropna(subset=[target_col, group_col])

    groups = data[group_col].astype(str)
    unique_groups = groups.nunique()
    actual_splits = min(n_splits, unique_groups)
    if actual_splits < 2:
        raise RuntimeError("Need at least two unique groups for GroupKFold.")

    gkf = GroupKFold(n_splits=actual_splits)

    rows = []
    X = data[categorical_cols + numeric_cols].copy()
    X = X.loc[:, ~pd.Index(X.columns).duplicated()].copy()
    y = data[target_col].to_numpy()

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        X_train = X.iloc[tr]
        X_test = X.iloc[te]
        y_train = y[tr]
        y_test = y[te]
        region_test = data.iloc[te]["region_type"].values

        pipe = make_model_pipeline(categorical_cols, numeric_cols, random_state=random_state)
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        rows.append(
            {
                "feature_set": feature_name,
                "fold": fold,
                "subset": "all",
                "n": len(y_test),
                "spearman": spearman_safe(y_test, pred),
                "rmse": rmse(y_test, pred),
                "r2": r2_score(y_test, pred),
            }
        )

        for subset in ["IDR", "ordered"]:
            mask = region_test == subset
            if mask.sum() < 5:
                continue
            rows.append(
                {
                    "feature_set": feature_name,
                    "fold": fold,
                    "subset": subset,
                    "n": int(mask.sum()),
                    "spearman": spearman_safe(y_test[mask], pred[mask]),
                    "rmse": rmse(y_test[mask], pred[mask]),
                    "r2": r2_score(y_test[mask], pred[mask]),
                }
            )

    return pd.DataFrame(rows)

# --- Plotting helpers ---

def save_distribution_plot(df: pd.DataFrame, outpath: Path) -> None:
    plt.figure(figsize=(7, 4.5))
    idr = df.loc[df["region_type"] == "IDR", "DMS_score_z"].dropna().values
    ordered = df.loc[df["region_type"] == "ordered", "DMS_score_z"].dropna().values

    bins = np.linspace(-4, 4, 50)
    plt.hist(ordered, bins=bins, alpha=0.6, density=True, label="ordered")
    plt.hist(idr, bins=bins, alpha=0.6, density=True, label="IDR")
    plt.xlabel("Assay-standardized mutation effect (DMS_score_z)")
    plt.ylabel("Density")
    plt.title("Mutation-effect distributions in ordered vs disordered regions")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()



def save_model_comparison_plot(cv_results: pd.DataFrame, outpath: Path, metric: str = "spearman") -> None:
    plot_df = cv_results[cv_results["subset"] == "all"].groupby("feature_set", as_index=False)[metric].mean()
    plot_df = plot_df.sort_values(metric, ascending=False)

    plt.figure(figsize=(7, 4.5))
    plt.bar(plot_df["feature_set"], plot_df[metric])
    plt.ylabel(metric)
    plt.title(f"Cross-validated model comparison ({metric}, all residues)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()



def save_subset_model_plot(cv_results: pd.DataFrame, outpath: Path, metric: str = "spearman") -> None:
    plot_df = cv_results.groupby(["feature_set", "subset"], as_index=False)[metric].mean()
    plot_df = plot_df[plot_df["subset"].isin(["IDR", "ordered"])]

    feature_sets = plot_df["feature_set"].unique().tolist()
    subsets = ["IDR", "ordered"]
    x = np.arange(len(feature_sets))
    width = 0.35

    plt.figure(figsize=(8, 4.8))
    for i, subset in enumerate(subsets):
        vals = []
        for fs in feature_sets:
            sub = plot_df[(plot_df["feature_set"] == fs) & (plot_df["subset"] == subset)]
            vals.append(sub[metric].iloc[0] if len(sub) else np.nan)
        plt.bar(x + (i - 0.5) * width, vals, width=width, label=subset)

    plt.xticks(x, feature_sets, rotation=20, ha="right")
    plt.ylabel(metric)
    plt.title(f"Model performance split by region type ({metric})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()


# -----------------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------------

## 5. Build the core benchmark and train baseline models

This section:
- loads ProteinGym and DisProt
- constructs the ProteinGym × DisProt overlap
- labels each mutation as **IDR** or **ordered**
- adds mutation descriptors
- trains the two baseline models used in the final analysis

The key outputs saved to Drive are:
- `analysis_table.csv`
- `region_summary.csv`
- `cv_results_by_fold.csv`
- `cv_summary.csv`


In [ ]:
# --- Core pipeline run ---

max_common = None if args.max_common_proteins is not None and args.max_common_proteins < 0 else args.max_common_proteins
dirs = ensure_dirs(args.project_root)

print("\n[1/8] Loading ProteinGym reference...")
ref = load_proteingym_reference(args.reference_csv)
print(f"Reference rows: {len(ref):,}")
print(f"Unique proteins in reference: {ref['UniProt_ID'].nunique():,}")

print("\n[2/8] Loading DisProt regions...")
disprot_regions = load_disprot_regions(args.disprot_file)
print(f"DisProt regions: {len(disprot_regions):,}")
print(f"Unique proteins in DisProt file: {disprot_regions['UniProt_ID'].nunique():,}")

print("\n[3/8] Building ProteinGym × DisProt intersection...")
merged = build_intersected_assay_table(
    reference_df=ref,
    disprot_regions=disprot_regions,
    assay_dir=args.assay_dir,
    max_common_proteins=max_common,
    min_variants_per_assay=args.min_variants_per_assay,
    max_assays_per_protein=args.max_assays_per_protein,
)
print(f"Variants after intersection/filtering: {len(merged):,}")
print(f"Assays kept: {merged['DMS_id'].nunique():,}")
print(f"Proteins kept: {merged['UniProt_ID'].nunique():,}")
if merged['UniProt_ID'].nunique() < 50:
    print("[WARN] Intersected protein count is <50. This is still usable, but keep scope modest.")

print("\n[4/8] Mapping ordered vs disordered residues...")
idr_masks = build_idr_masks(merged[["UniProt_ID", "target_seq"]].drop_duplicates(), disprot_regions)
merged = add_region_features(merged, idr_masks=idr_masks, seq_window=args.seq_window)
merged = add_mutation_descriptor_features(merged)
print(merged["region_type"].value_counts(dropna=False))

print("\n[5/8] Adding optional structure-derived features...")
structure_cache = build_structure_feature_cache(
    merged,
    structure_dir=args.structure_dir,
    seq_window=args.seq_window,
    struct_radius=args.struct_radius,
)
merged = add_structure_features(merged, structure_cache)
print("Structure available for variants:", int(merged["structure_available"].sum()), "of", len(merged))

analysis_df = merged[merged["region_type"].isin(["IDR", "ordered"])].copy()
analysis_df = analysis_df.loc[:, ~pd.Index(analysis_df.columns).duplicated()].copy()
if analysis_df.empty:
    raise RuntimeError("No variants with usable IDR/ordered assignments were found after mapping.")

analysis_df.to_csv(dirs["table_dir"] / "analysis_table.csv", index=False)

region_summary = (
    analysis_df.groupby("region_type")
    .agg(
        n_variants=("mutant", "size"),
        n_proteins=("UniProt_ID", "nunique"),
        mean_DMS_score_z=("DMS_score_z", "mean"),
        median_DMS_score_z=("DMS_score_z", "median"),
        mean_plddt=("plddt", "mean"),
    )
    .reset_index()
)
region_summary.to_csv(dirs["table_dir"] / "region_summary.csv", index=False)

print("\n[6/8] Training and evaluating models...")
feature_sets = {
    "mutation_only": {
        "cat": ["wt_aa", "mut_aa"],
        "num": [
            "position_frac", "delta_hydropathy", "delta_volume", "delta_charge",
            "wt_is_aromatic", "mut_is_aromatic", "wt_is_polar", "mut_is_polar",
            "wt_is_small", "mut_is_small", "is_conservative_substitution",
        ],
    },
    "mutation_plus_region": {
        "cat": ["wt_aa", "mut_aa", "region_type"],
        "num": [
            "position_frac", "local_disorder_density",
            "delta_hydropathy", "delta_volume", "delta_charge",
            "wt_is_aromatic", "mut_is_aromatic", "wt_is_polar", "mut_is_polar",
            "wt_is_small", "mut_is_small", "is_conservative_substitution",
        ],
    },
}

cv_tables = []
for name, cols in feature_sets.items():
    print(f"  - Evaluating {name} ...")
    res = evaluate_feature_set(
        analysis_df,
        feature_name=name,
        categorical_cols=cols["cat"],
        numeric_cols=cols["num"],
        target_col=args.target_column,
        group_col=args.group_column,
        n_splits=args.n_splits,
        random_state=DEFAULT_RANDOM_STATE,
    )
    cv_tables.append(res)

cv_results = pd.concat(cv_tables, ignore_index=True)
cv_results.to_csv(dirs["table_dir"] / "cv_results_by_fold.csv", index=False)

cv_summary = (
    cv_results.groupby(["feature_set", "subset"], as_index=False)
    .agg(
        mean_spearman=("spearman", "mean"),
        std_spearman=("spearman", "std"),
        mean_rmse=("rmse", "mean"),
        mean_r2=("r2", "mean"),
        mean_n=("n", "mean"),
    )
    .sort_values(["subset", "mean_spearman"], ascending=[True, False])
)
cv_summary.to_csv(dirs["table_dir"] / "cv_summary.csv", index=False)

print("\n[7/8] Saving figures...")
save_distribution_plot(analysis_df, dirs["fig_dir"] / "idr_vs_ordered_dms_score_distribution.png")
save_model_comparison_plot(cv_results, dirs["fig_dir"] / "model_comparison_spearman.png", metric="spearman")
save_subset_model_plot(cv_results, dirs["fig_dir"] / "model_comparison_by_region_spearman.png", metric="spearman")

print("\n[8/8] Done.")
print("\nTop-level outputs:")
print(f"  Analysis table : {dirs['table_dir'] / 'analysis_table.csv'}")
print(f"  Region summary : {dirs['table_dir'] / 'region_summary.csv'}")
print(f"  CV results     : {dirs['table_dir'] / 'cv_results_by_fold.csv'}")
print(f"  CV summary     : {dirs['table_dir'] / 'cv_summary.csv'}")
print(f"  Figures        : {dirs['fig_dir']}")

display(region_summary)
display(cv_summary)


## 6. Region-level descriptive analysis

This section tests whether the **distribution of assay-standardized mutation effects** differs between IDRs and ordered regions.


In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu, ks_2samp

idr = analysis_df.loc[analysis_df["region_type"] == "IDR", "DMS_score_z"].dropna().to_numpy()
ordr = analysis_df.loc[analysis_df["region_type"] == "ordered", "DMS_score_z"].dropna().to_numpy()

def cohens_d(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    pooled = np.sqrt(((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled

def cliffs_delta(x, y, cap=5000):
    x = np.asarray(x)[:cap]
    y = np.asarray(y)[:cap]
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    return (gt - lt) / (len(x) * len(y))

mw = mannwhitneyu(idr, ordr, alternative="two-sided")
ks = ks_2samp(idr, ordr)

print("IDR mean:", idr.mean())
print("Ordered mean:", ordr.mean())
print("IDR median:", np.median(idr))
print("Ordered median:", np.median(ordr))
print("Mean difference (IDR - ordered):", idr.mean() - ordr.mean())
print("Median difference (IDR - ordered):", np.median(idr) - np.median(ordr))
print("Mann–Whitney U p-value:", mw.pvalue)
print("KS test p-value:", ks.pvalue)
print("Cohen's d:", cohens_d(idr, ordr))
print("Cliff's delta (approx):", cliffs_delta(idr, ordr))


## 7. Protein-level heterogeneity

The pooled benchmark may be dominated by a few large assays.  
This section asks whether the **IDR-vs-ordered difference is consistent across proteins**.


In [ ]:
from scipy.stats import wilcoxon

protein_region = (
    analysis_df
    .groupby(["UniProt_ID", "region_type"])["DMS_score_z"]
    .median()
    .unstack()
)

protein_region = protein_region.dropna(subset=["IDR", "ordered"]).copy()
protein_region["IDR_minus_ordered"] = protein_region["IDR"] - protein_region["ordered"]

print("Proteins with both region types:", len(protein_region))
print("Proteins with positive IDR - ordered difference:", int((protein_region["IDR_minus_ordered"] > 0).sum()))
print("Proteins with negative IDR - ordered difference:", int((protein_region["IDR_minus_ordered"] < 0).sum()))
print(protein_region["IDR_minus_ordered"].describe())

if len(protein_region) >= 2:
    w = wilcoxon(protein_region["IDR"], protein_region["ordered"])
    print("Wilcoxon paired p-value:", w.pvalue)

display(protein_region.sort_values("IDR_minus_ordered", ascending=False))

plt.figure(figsize=(8, 4))
plt.axhline(0, linestyle="--")
plt.bar(range(len(protein_region)), protein_region["IDR_minus_ordered"].sort_values().values)
plt.title("Per-protein median DMS_score_z difference (IDR - ordered)")
plt.xlabel("Proteins")
plt.ylabel("Median difference")
plt.tight_layout()
plt.show()


## 8. Balanced subsampling robustness check

Because the benchmark contains many more ordered mutations than IDR mutations, this section checks whether the pooled shift survives **size-matched resampling**.


In [ ]:
idr_df = analysis_df[analysis_df["region_type"] == "IDR"].copy()
ord_df = analysis_df[analysis_df["region_type"] == "ordered"].copy()

n_idr = len(idr_df)
rng = np.random.default_rng(42)

mean_diffs = []
median_diffs = []

for _ in range(200):
    ord_sample = ord_df.sample(n=n_idr, replace=False, random_state=int(rng.integers(1_000_000)))
    mean_diffs.append(idr_df["DMS_score_z"].mean() - ord_sample["DMS_score_z"].mean())
    median_diffs.append(idr_df["DMS_score_z"].median() - ord_sample["DMS_score_z"].median())

print("Balanced mean diff: mean =", np.mean(mean_diffs), "std =", np.std(mean_diffs))
print("Balanced median diff: mean =", np.mean(median_diffs), "std =", np.std(median_diffs))
print("95% interval for mean diff:", np.percentile(mean_diffs, [2.5, 97.5]))
print("95% interval for median diff:", np.percentile(median_diffs, [2.5, 97.5]))

plt.figure(figsize=(7, 4))
plt.hist(mean_diffs, bins=30)
plt.axvline(0, linestyle="--")
plt.title("Balanced subsampling: IDR - ordered mean DMS_score_z")
plt.xlabel("Mean difference")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 9. Cross-validated model comparison and out-of-fold error analysis

These cells summarize whether:
- adding the **region label** improves prediction
- prediction quality differs between **IDR** and **ordered** residues
- absolute error and residual behavior differ by region


In [ ]:
display(cv_summary)

from IPython.display import Image, display as ipy_display

for fname in [
    "idr_vs_ordered_dms_score_distribution.png",
    "model_comparison_spearman.png",
    "model_comparison_by_region_spearman.png",
]:
    path = dirs["fig_dir"] / fname
    if path.exists():
        print(path.name)
        ipy_display(Image(filename=str(path)))


In [ ]:
from sklearn.model_selection import GroupKFold

def get_oof_predictions(
    df,
    categorical_cols,
    numeric_cols,
    target_col,
    group_col,
    n_splits,
    random_state,
):
    use_cols = categorical_cols + numeric_cols + [target_col, group_col, "region_type", "UniProt_ID", "mutant"]
    use_cols = list(dict.fromkeys(use_cols))
    data = df[use_cols].dropna(subset=[target_col, group_col]).copy()

    groups = data[group_col].astype(str)
    actual_splits = min(n_splits, groups.nunique())
    gkf = GroupKFold(n_splits=actual_splits)

    X = data[categorical_cols + numeric_cols].copy()
    y = data[target_col].to_numpy()
    preds = np.full(len(data), np.nan)

    for tr, te in gkf.split(X, y, groups=groups):
        pipe = make_model_pipeline(categorical_cols, numeric_cols, random_state=random_state)
        pipe.fit(X.iloc[tr], y[tr])
        preds[te] = pipe.predict(X.iloc[te])

    out = data.copy()
    out["y_true"] = y
    out["y_pred"] = preds
    out["residual"] = out["y_true"] - out["y_pred"]
    out["abs_error"] = np.abs(out["residual"])
    return out

oof_df = get_oof_predictions(
    analysis_df,
    categorical_cols=feature_sets["mutation_plus_region"]["cat"],
    numeric_cols=feature_sets["mutation_plus_region"]["num"],
    target_col=args.target_column,
    group_col=args.group_column,
    n_splits=args.n_splits,
    random_state=DEFAULT_RANDOM_STATE,
)

display(
    oof_df.groupby("region_type")[["abs_error", "residual"]]
    .agg(["mean", "median", "std"])
)

tmp = oof_df.groupby("region_type")["abs_error"].mean().reindex(["IDR", "ordered"])

plt.figure(figsize=(5, 4))
plt.bar(tmp.index, tmp.values)
plt.title("Mean absolute prediction error by region")
plt.ylabel("Absolute error")
plt.tight_layout()
plt.show()


## 10. Mutation-class exploratory analysis

This optional section checks whether a simple mutation class — here, **charge-changing vs non-charge-changing** substitutions — behaves differently across region types.


In [ ]:
analysis_df["charge_changing"] = (analysis_df["delta_charge"] != 0).astype(int)

charge_summary = (
    analysis_df
    .groupby(["region_type", "charge_changing", "is_conservative_substitution"])["DMS_score_z"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

display(charge_summary)
display(analysis_df.groupby(["region_type", "charge_changing"]).size().rename("n").reset_index())

pivot_mean = (
    analysis_df
    .groupby(["region_type", "charge_changing"])["DMS_score_z"]
    .mean()
    .unstack()
)

display(pivot_mean)

pivot_mean.plot(kind="bar", figsize=(6, 4))
plt.title("Mean DMS_score_z by region and charge-changing status")
plt.ylabel("Mean DMS_score_z")
plt.tight_layout()
plt.show()


## 11. Final interpretation notes

Use the cells above to support a final summary along these lines:

- The ProteinGym × DisProt overlap is **modest but usable**.
- IDR mutations appear **somewhat more tolerated on average** than ordered-region mutations in the pooled benchmark.
- The pooled trend survives **balanced subsampling**, so it is not just a sample-size artifact.
- The protein-level effect is **heterogeneous**, not universal across all proteins.
- Predictive performance is **lower in IDRs** than in ordered regions when evaluated by cross-validated Spearman correlation.
- Adding a simple `region_type` feature gives **little improvement** over mutation-only features.
- Structure-based conclusions should be phrased carefully if `structure_available == 0`.
